# Codeforces Rating Prediction

This notebook is split into small runnable cells. Set `QUICK_MODE = False` for a fuller TF-IDF model, and set `RUN_GRID_SEARCH = True` only when you want to spend extra time tuning Gradient Boosting.


## 1. Setup


In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

from feature_engineering import add_structural_features

DATA_FILE = PROJECT_DIR / "data" / "cleaned" / "codeforces_cleaned.csv"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

QUICK_MODE = True
RUN_GRID_SEARCH = False
RANDOM_STATE = 42

TFIDF_MAX_FEATURES = 5000 if QUICK_MODE else 10000

STRUCTURAL_COLUMNS = [
    "char_count",
    "word_count",
    "sentence_count",
    "number_count",
    "math_symbol_count",
    "avg_word_length",
]

def rmse_score(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def evaluate_model(name, model, X_test, y_test):
    predictions = model.predict(X_test)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": rmse_score(y_test, predictions),
        "R2": r2_score(y_test, predictions),
    }

def parse_tags(tags):
    return [
        tag.strip()
        for tag in str(tags).split(";")
        if tag.strip()
    ]


## 2. Load Data


In [ ]:
df = pd.read_csv(DATA_FILE)
df["statement"] = df["statement"].fillna("")
df["tags"] = df["tags"].fillna("")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print(df["rating"].describe())

df.head()


## 3. Train/Test Split and TF-IDF


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

y_train = train_df["rating"]
y_test = test_df["rating"]

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words="english",
    ngram_range=(1, 2),
)

X_train_text = tfidf.fit_transform(train_df["statement"])
X_test_text = tfidf.transform(test_df["statement"])

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("TF-IDF shape:", X_train_text.shape)


## 4. Experiment 1: Statement Text Only


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_text, y_train)

ridge_text_model = Ridge(alpha=1.0)
ridge_text_model.fit(X_train_text, y_train)

experiment_1_results = pd.DataFrame([
    evaluate_model("Linear Regression", linear_model, X_test_text, y_test),
    evaluate_model("Ridge Regression", ridge_text_model, X_test_text, y_test),
])

experiment_1_results.to_csv(
    RESULTS_DIR / "experiment_01_statement_only.csv",
    index=False,
)
experiment_1_results.to_csv(
    RESULTS_DIR / "model_results.csv",
    index=False,
)

joblib.dump(ridge_text_model, MODELS_DIR / "ridge_rating_model.pkl")
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.pkl")

experiment_1_results


## 5. Add Structural Features


In [ ]:
df_features = add_structural_features(df)
train_features_df, test_features_df = train_test_split(
    df_features,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_struct = scaler.fit_transform(train_features_df[STRUCTURAL_COLUMNS])
X_test_struct = scaler.transform(test_features_df[STRUCTURAL_COLUMNS])

print(df_features[STRUCTURAL_COLUMNS + ["rating"]].corr()["rating"].sort_values(ascending=False))


## 6. Experiment 2: Text + Structural Features


In [ ]:
X_train_exp2 = hstack([X_train_text, X_train_struct])
X_test_exp2 = hstack([X_test_text, X_test_struct])

ridge_exp2 = Ridge(alpha=1.0)
ridge_exp2.fit(X_train_exp2, y_train)

experiment_2_results = pd.DataFrame([
    {
        "Model": "Ridge Regression",
        "Features": "TF-IDF + Structural",
        **evaluate_model("Ridge Regression", ridge_exp2, X_test_exp2, y_test),
    }
])
experiment_2_results = experiment_2_results[["Model", "Features", "MAE", "RMSE", "R2"]]
experiment_2_results.to_csv(
    RESULTS_DIR / "experiment_02_structural_features.csv",
    index=False,
)

experiment_2_results


## 7. Add Tag Features


In [ ]:
train_tags = train_features_df["tags"].apply(parse_tags)
test_tags = test_features_df["tags"].apply(parse_tags)

mlb = MultiLabelBinarizer()
X_train_tags = csr_matrix(mlb.fit_transform(train_tags))
X_test_tags = csr_matrix(mlb.transform(test_tags))

print("Number of tag features:", len(mlb.classes_))
print("Tag matrix:", X_train_tags.shape)


## 8. Experiment 3: Text + Structural + Tags


In [ ]:
X_train_exp3 = hstack([X_train_text, X_train_struct, X_train_tags])
X_test_exp3 = hstack([X_test_text, X_test_struct, X_test_tags])

ridge_exp3 = Ridge(alpha=1.0)
ridge_exp3.fit(X_train_exp3, y_train)

experiment_3_results = pd.DataFrame([
    {
        "Model": "Ridge Regression",
        "Features": "TF-IDF + Structural + Tags",
        **evaluate_model("Ridge Regression", ridge_exp3, X_test_exp3, y_test),
    }
])
experiment_3_results = experiment_3_results[["Model", "Features", "MAE", "RMSE", "R2"]]
experiment_3_results.to_csv(
    RESULTS_DIR / "experiment_03_tags.csv",
    index=False,
)

experiment_3_results


## 9. Experiment 4: Fast Model Comparison


In [ ]:
X_train_dense = X_train_exp3.toarray()
X_test_dense = X_test_exp3.toarray()

gb_model = GradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    n_estimators=150 if QUICK_MODE else 300,
    random_state=RANDOM_STATE,
)
gb_model.fit(X_train_dense, y_train)

experiment_4_results = pd.DataFrame([
    {
        "Model": "Ridge Regression",
        "Features": "TF-IDF + Structural + Tags",
        **evaluate_model("Ridge Regression", ridge_exp3, X_test_exp3, y_test),
    },
    {
        "Model": "Gradient Boosting",
        "Features": "TF-IDF + Structural + Tags",
        **evaluate_model("Gradient Boosting", gb_model, X_test_dense, y_test),
    },
])
experiment_4_results = experiment_4_results[["Model", "Features", "MAE", "RMSE", "R2"]]
experiment_4_results.to_csv(
    RESULTS_DIR / "experiment_04_model_comparison.csv",
    index=False,
)

experiment_4_results.sort_values("MAE")


## 10. Optional Hyperparameter Tuning


In [ ]:
if RUN_GRID_SEARCH:
    param_grid = {
        "n_estimators": [100, 200] if QUICK_MODE else [100, 200, 300],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "min_samples_leaf": [3, 5],
    }

    grid_search = GridSearchCV(
        estimator=GradientBoostingRegressor(random_state=RANDOM_STATE),
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=3,
        n_jobs=-1,
        verbose=1,
    )
    grid_search.fit(X_train_dense, y_train)
    best_gb_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
else:
    best_gb_model = gb_model
    best_params = gb_model.get_params()

best_gb_metrics = evaluate_model(
    "Gradient Boosting Tuned" if RUN_GRID_SEARCH else "Gradient Boosting",
    best_gb_model,
    X_test_dense,
    y_test,
)

experiment_5_results = pd.DataFrame([{
    "Model": best_gb_metrics["Model"],
    "Features": "TF-IDF + Structural + Tags",
    "MAE": best_gb_metrics["MAE"],
    "RMSE": best_gb_metrics["RMSE"],
    "R2": best_gb_metrics["R2"],
}])
experiment_5_results.to_csv(
    RESULTS_DIR / "experiment_05_hyperparameter_tuning.csv",
    index=False,
)

with open(RESULTS_DIR / "best_gradient_boosting_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

experiment_5_results


## 11. Save Final Model and Preprocessors


In [ ]:
candidate_models = [
    ("ridge", ridge_exp3, X_test_exp3, evaluate_model("Ridge Regression", ridge_exp3, X_test_exp3, y_test)),
    ("gradient_boosting", best_gb_model, X_test_dense, best_gb_metrics),
]

best_name, final_model, _, final_metrics = min(
    candidate_models,
    key=lambda item: item[3]["MAE"],
)

joblib.dump(final_model, MODELS_DIR / "final_rating_model.pkl")
joblib.dump(tfidf, MODELS_DIR / "final_tfidf_vectorizer.pkl")
joblib.dump(scaler, MODELS_DIR / "structural_scaler.pkl")
joblib.dump(mlb, MODELS_DIR / "tag_encoder.pkl")

print("Saved final model:", best_name)
print(final_metrics)


## 12. Quick Prediction Test


In [ ]:
sample_statement = """
You are given an array of n integers. Find the maximum possible sum after performing operations.
"""
sample_tags = "greedy;dp"

sample_df = pd.DataFrame({
    "statement": [sample_statement],
    "tags": [sample_tags],
})
sample_df = add_structural_features(sample_df)

sample_text = tfidf.transform(sample_df["statement"])
sample_struct = scaler.transform(sample_df[STRUCTURAL_COLUMNS])
sample_tag_list = [[tag for tag in parse_tags(sample_tags) if tag in set(mlb.classes_)]]
sample_tags_encoded = csr_matrix(mlb.transform(sample_tag_list))
sample_features = hstack([sample_text, sample_struct, sample_tags_encoded])

if best_name == "gradient_boosting":
    sample_features = sample_features.toarray()

predicted_rating = final_model.predict(sample_features)[0]
print("Predicted Rating:", int(round(predicted_rating)))
